### Miguel Baños Baladrón
### Miguel Pérez Francos
### Rodrigo Touceda Tapias

# Importaciones

In [12]:
using CSV, DataFrames, Glob, Statistics, Random, MLJ

# Preparación de los datos (20%)

## 1. Carga y unificación de los datos

In [3]:
base = "Datos Práctica"

# CSV del Investigador A
csv_inv_a = glob("Investigador A/day */*.csv", base)

# CSV del Investigador B
csv_inv_b = glob("Investigador B/*.csv", base)

all_csv = vcat(csv_inv_a, csv_inv_b)

dfs = [CSV.read(file, DataFrame) for file in all_csv]
df_total = vcat(dfs...)

println("Dataset correctamente cargado y unificado.")
println("Número de variables:         ", ncol(df_total))
println("Número de instancias:        ", nrow(df_total))
println("Número de individuos:        ", length(unique(df_total.subject)))
println("Número de clases de salida:  ", length(unique(df_total.Activity)))

Dataset correctamente cargado y unificado.
Número de variables:         563
Número de instancias:        10299
Número de individuos:        30
Número de clases de salida:  6


In [4]:
println("Clases:\n")
for activity in unique(df_total.Activity)
    println(activity)
end

Clases:

STANDING
SITTING
LAYING
WALKING
WALKING_UPSTAIRS
WALKING_DOWNSTAIRS


# 2. Análisis de valores ausentes

In [5]:
# Porcentajes de nulos por variable
nulos_por_variable = DataFrame(
    Variable = names(df_total),
    PorcentajeNulos = [count(ismissing, df_total[!, col]) / nrow(df_total) * 100 for col in names(df_total)]
)

# Porcentaje de nulos en todo el dataset
total_nulos = sum(count(ismissing, df_total[!, col]) for col in names(df_total))
total_valores = nrow(df_total) * ncol(df_total)

porcentaje_total_nulos = (total_nulos / total_valores) * 100

println("Porcentaje total de valores nulos en el dataset: $(porcentaje_total_nulos)%")

Porcentaje total de valores nulos en el dataset: 0.9984242033534787%


# 3. Tratamiento y transformación de datos

In [6]:
n = nrow(df_total)

res = DataFrame(variable = String[], n_missing = Int[], porcentaje = Float64[])

for col in names(df_total)
    n_miss = count(ismissing, df_total[!, col])
    porc = round((n_miss / n) * 100, digits=2)
    push!(res, (string(col), n_miss, porc))
end

# Ordenar de mayor a menor porcentaje
sort!(res, :porcentaje, rev=true)

# Mostrar solo las 10 primeras
first(res, 3)

Row,variable,n_missing,porcentaje
,String,Int64,Float64
1,tBodyGyroMag-mad(),1033,10.03
2,tBodyGyroMag-iqr(),1033,10.03
3,fBodyAcc-mad()-Y,1032,10.02


# Como la variable con mayor porcentaje de nulos no presenta un valor muy alto, decidimos imputar todas las variables

In [7]:
# Rellenamos con la mediana los valores nulos porque es menos sensible a outliers
df_imputado = deepcopy(df_total)
gdf = groupby(df_imputado, :subject) # Agrupamos por individuo para que cada dato nulo se rellene con la mediana de los valores de dicho individuo.

for subdf in gdf
    for col in names(subdf)
        if col in (:subject, :activity)
            continue
        end

        # Ignoramos variables no numéricas
        coldata = subdf[!, col]
        if !(eltype(skipmissing(coldata)) <: Number)
            continue
        end

        mediana = median(skipmissing(coldata))
        replace!(coldata, missing => mediana)
    end
end

# DataFrame sin la variable 'subject' para entrenamiento
df_without_subject = select(df_imputado, Not(:subject));

# 4. Partición Holdout

In [15]:
# Lista de sujetos únicos
subjects = unique(df_imputado.subject)

# Fijamos semilla 104
Random.seed!(104)

# Barajamos el orden de los sujetos
shuffle!(subjects)

# 10% de sujetos para test
n_test = round(Int, length(subjects) * 0.10)

test_subjects      = subjects[1:n_test]
trainval_subjects  = subjects[n_test+1:end]

println("Sujetos en TEST: ", sort(test_subjects))
println("Sujetos en TRAIN+VAL: ", sort(trainval_subjects))

df_trainval = filter(row -> row.subject in trainval_subjects, df_imputado);
df_test     = filter(row -> row.subject in test_subjects, df_imputado);

Sujetos en TEST: [18, 22, 25]
Sujetos en TRAIN+VAL: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 23, 24, 26, 27, 28, 29, 30]


# 5. Cross-Validation individual wise

In [16]:
features_cols = names(df_trainval, Not([:subject, :Activity]))
X = Matrix(df_trainval[:, features_cols])
y = df_trainval.Activity
groups = df_trainval.subject

Random.seed!(104)

unique_subjects = unique(groups)
shuffled = shuffle(unique_subjects)
n_subj = length(unique_subjects)

k = 5
# Crear folds 
fold_ids = [mod1(i,k) for i in 1:n_subj] # mod1 para que los índices vayande 1 a k 
folds_subjects = [shuffled[fold_ids .== f] for f in 1:k] # Lista de sujetos por fold

for (i, test_subj_fold) in enumerate(folds_subjects)
    println("\n=== Fold $i ===")

    # Índices de validación y entrenamiento
    val_idx  = findall(x -> x in test_subj_fold, groups)
    train_idx = findall(x -> !(x in test_subj_fold), groups)

    # Mostrar info útil
    println("Sujetos en VALIDACIÓN: ", sort(test_subj_fold))
    println("Nº instancias TRAIN: ", length(train_idx))
    println("Nº instancias VAL:   ", length(val_idx))
end


=== Fold 1 ===
Sujetos en VALIDACIÓN: [11, 12, 15, 16, 20, 23]
Nº instancias TRAIN: 7149
Nº instancias VAL:   2056

=== Fold 2 ===
Sujetos en VALIDACIÓN: [1, 4, 13, 21, 24, 27]
Nº instancias TRAIN: 7049
Nº instancias VAL:   2156

=== Fold 3 ===
Sujetos en VALIDACIÓN: [2, 3, 8, 10, 28]
Nº instancias TRAIN: 7605
Nº instancias VAL:   1600

=== Fold 4 ===
Sujetos en VALIDACIÓN: [6, 7, 14, 19, 26]
Nº instancias TRAIN: 7497
Nº instancias VAL:   1708

=== Fold 5 ===
Sujetos en VALIDACIÓN: [5, 9, 17, 29, 30]
Nº instancias TRAIN: 7520
Nº instancias VAL:   1685
